La prima cosa che faccio è prendere i problemi singoli di ogni file e caricarli separatamente

In [ ]:
import os
import glob

documents_dir = os.path.join("..", "Documents")

problems = []
for filepath in sorted(glob.glob(os.path.join(documents_dir, "*.txt"))):
    year = os.path.basename(filepath).split("_")[0]
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    for chunk in content.split("-%-"):
        problem = chunk.strip()
        if problem:
            problems.append({"year": year, "source_file": os.path.basename(filepath), "text": problem})

print(f"Totale problemi estratti: {len(problems)}")


In [ ]:
#10 problemi più lunghi
sorted_problems = sorted(problems, key=lambda x: len(x["text"]), reverse=True)
top_10_problems = sorted_problems[:10]
print("\nI 10 problemi più lunghi:")
for i, problem in enumerate(top_10_problems, start=1):
    print(f"{i}. Lunghezza: {len(problem['text'])}, Anno: {problem['year']}, File: {problem['source_file']}")
    print(f"   Testo: {problem['text']}\n") 

# Caricamento su qdrant

Eliminazione dataset

In [ ]:
from qdrant_client import QdrantClient

COLLECTION_NAME = "Tesina_BI"

qdrant = QdrantClient(host="host.docker.internal", port=6333)

if qdrant.collection_exists(COLLECTION_NAME):
    qdrant.delete_collection(COLLECTION_NAME)
    print(f"Collezione '{COLLECTION_NAME}' eliminata.")
else:
    print(f"Collezione '{COLLECTION_NAME}' non esisteva, nulla da eliminare.")


Ingestion da 0

In [ ]:
from qdrant_client import QdrantClient
from llama_index.core import Settings, StorageContext, VectorStoreIndex
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore

COLLECTION_NAME = "Tesina_BI"

print("Connessione a Qdrant...")
qdrant = QdrantClient(host="host.docker.internal", port=6333)
print("Connesso. Collezioni esistenti:", [c.name for c in qdrant.get_collections().collections])

print("Caricamento embedding model BGE-M3 (fp16)...")
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    model_kwargs={"torch_dtype": "float16"},
)
print("Embedding model caricato.")

print("Creazione vector store ibrido (dense BGE-M3 + sparse BM25)...")
vector_store = QdrantVectorStore(
    collection_name=COLLECTION_NAME,
    client=qdrant,
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25",
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
print("Vector store pronto.")

nodes = [
    TextNode(text=p["text"], metadata={"year": p["year"], "source_file": p["source_file"]})
    for p in problems
]
print(f"Costruiti {len(nodes)} nodi.")

print("Creazione indice (vuoto, la collezione su Qdrant nasce al primo insert)...")
index = VectorStoreIndex([], storage_context=storage_context)

print("Inserimento nodi uno alla volta...")
for i, node in enumerate(nodes, start=1):
    index.insert_nodes([node])
    print(f"[{i}/{len(nodes)}] caricato ({node.metadata['source_file']}): {node.text[:60]!r}")

print("Fatto. Collezioni ora presenti:", [c.name for c in qdrant.get_collections().collections])


Connessione singola a Qdrant a vdb già creato

In [ ]:
from qdrant_client import QdrantClient
from llama_index.core import Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore

COLLECTION_NAME = "Tesina_BI"

print("Connessione a Qdrant...")
qdrant = QdrantClient(host="host.docker.internal", port=6333)
print("Connesso. Collezioni esistenti:", [c.name for c in qdrant.get_collections().collections])

print("Caricamento embedding model BGE-M3 (fp16)... (serve solo per embeddare le query in fase di retrieval)")
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3",
    model_kwargs={"torch_dtype": "float16"},
)
print("Embedding model caricato.")

vector_store = QdrantVectorStore(
    collection_name=COLLECTION_NAME,
    client=qdrant,
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25",
)

# la collezione esiste già su Qdrant con tutti i punti: mi ci aggancio senza reinserire nulla
index = VectorStoreIndex.from_vector_store(vector_store)

collection_info = qdrant.get_collection(COLLECTION_NAME)
print(f"Riconnesso a '{COLLECTION_NAME}': {collection_info.points_count} punti già presenti.")


# LLM 

prompt: 
Due problemi sono "simili" se condividono almeno due di questi tre elementi: stesso sistema/componente coinvolto, stessa causa, stessa soluzione.

In [ ]:
from llama_index.llms.openai_like import OpenAILike

llm = OpenAILike(
    model="google/gemma-3-12b",  
    api_base="http://host.docker.internal:1234/v1",
    api_key="lm-studio",  
    is_chat_model=True,
    is_local=True,
    context_window=16384,
    temperature=0.2,
    timeout=300.0,  
    max_tokens=4096,  
)

response = llm.complete("Ciao, funziona?")
print(response)


Struttura UnionFind

In [ ]:
class UnionFind:
    def __init__(self, n, verbose=True):
        self.parent = list(range(n))
        self.verbose = verbose

    def find(self, x):
        original = x
        path = [x]
        while self.parent[x] != x:
            x = self.parent[x]
            path.append(x)
        root = x
        # path compression: ricollego ogni nodo attraversato direttamente alla radice
        for node in path[:-1]:
            if self.verbose and self.parent[node] != root:
                print(f"      [find] path compression: {node} punta ora direttamente a {root} (prima puntava a {self.parent[node]})")
            self.parent[node] = root
        if self.verbose:
            print(f"      [find] find({original}) = {root}  (percorso attraversato: {path})")
        return root

    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx == ry:
            if self.verbose:
                print(f"      [union] {x} e {y} sono già nello stesso gruppo (radice {rx}), nulla da fare")
            return
        self.parent[rx] = ry
        if self.verbose:
            print(f"      [union] fondo il gruppo con radice {rx} dentro quello con radice {ry}")


SIMILARITY_PROMPT = """Due problemi sono "simili" se condividono almeno due di questi tre elementi: stesso sistema/componente coinvolto, stessa causa, stessa soluzione.

Problema A:
{problem_a}

Problema B:
{problem_b}

Sono lo stesso problema secondo questa definizione? Rispondi SOLO con "SI" o "NO", senza spiegazioni.
"""


def are_same_problem(text_a, text_b):
    prompt = SIMILARITY_PROMPT.format(problem_a=text_a, problem_b=text_b)
    try:
        response = llm.complete(prompt)
    except Exception as e:
        print(f"      [are_same_problem] ERRORE chiamata LLM ({e!r}), considero come 'diverso' e proseguo")
        return False
    answer = str(response).strip().upper()
    return answer.startswith("SI")


# mappa testo -> indice in problems, per riportare i risultati del retriever
# alla lista originale (i nodi sono stati inseriti con lo stesso testo, invariato)
text_to_idx = {p["text"]: i for i, p in enumerate(problems)}


Test singolo

In [ ]:
# Test rapido: verifico a mano l'output del modello su un caso "probabilmente simile"
# (il top-1 restituito da Qdrant per lo stesso problema) e uno "probabilmente diverso"
# (un problema preso a caso e lontano nella lista), prima di lanciare il loop completo.

test_retriever = index.as_retriever(vector_store_query_mode="hybrid", similarity_top_k=2)

problem_a = problems[0]["text"]
candidate = test_retriever.retrieve(problem_a)[1]  # [0] è probabilmente se stesso, prendo il secondo
problem_b_simile = candidate.node.text

problem_b_diverso = problems[200]["text"]

print("=== Caso probabilmente SIMILE ===")
print("A:", problem_a, "...")
print("B:", problem_b_simile, "...")
raw_response = llm.complete(SIMILARITY_PROMPT.format(problem_a=problem_a, problem_b=problem_b_simile))
print("Risposta grezza del modello:", repr(str(raw_response)))
print("Interpretata come:", are_same_problem(problem_a, problem_b_simile))

print()
print("=== Caso probabilmente DIVERSO ===")
print("A:", problem_a, "...")
print("B:", problem_b_diverso, "...")
raw_response = llm.complete(SIMILARITY_PROMPT.format(problem_a=problem_a, problem_b=problem_b_diverso))
print("Risposta grezza del modello:", repr(str(raw_response)))
print("Interpretata come:", are_same_problem(problem_a, problem_b_diverso))


Controllo similarità + llm

In [ ]:
retriever = index.as_retriever( #6 perché ne prendo 1 che è sé stesso (si spera)
    vector_store_query_mode="hybrid",
    similarity_top_k=6,
    sparse_top_k=6,
    hybrid_top_k=6,
)

uf = UnionFind(len(problems))
checked_pairs = set()  # coppie (i, j) già valutate dall'LLM, a prescindere dall'esito

for i, problem in enumerate(problems):
    print(f"[{i + 1}/{len(problems)}] Analizzo: {problem['text'][:60]!r}")
    results = retriever.retrieve(problem["text"])
    for res in results:
        j = text_to_idx.get(res.node.text)
        if j is None or j == i:
            continue
        if uf.find(i) == uf.find(j):
            print(f"    vs [{j}] score={res.score:.3f} -> già nello stesso gruppo, skip")
            continue
        pair_key = (min(i, j), max(i, j))
        if pair_key in checked_pairs:
            print(f"    vs [{j}] score={res.score:.3f} -> coppia già valutata (esito: diverso), skip")
            continue
        checked_pairs.add(pair_key)
        same = are_same_problem(problem["text"], res.node.text)
        print(f"    vs [{j}] score={res.score:.3f} -> {'STESSO PROBLEMA' if same else 'diverso'}")
        if same:
            uf.union(i, j)

print("Fatto.")


(circa 1700 chiamate a LLM)

Risultati

In [ ]:
from collections import defaultdict

clusters = defaultdict(list)
for i in range(len(problems)):
    clusters[uf.find(i)].append(i)

merged_groups = [members for members in clusters.values() if len(members) > 1]
singletons = [members[0] for members in clusters.values() if len(members) == 1]

print(f"Problemi totali: {len(problems)}")
print(f"Gruppi di duplicati trovati: {len(merged_groups)}")
print(f"Problemi rimasti unici: {len(singletons)}")

for group in merged_groups:
    print("---")
    for idx in group:
        print(f"  [{idx}] ({problems[idx]['source_file']}) {problems[idx]['text'][:80]!r}")


Salvataggio risultati

In [ ]:
output_path = os.path.join("..", "Documents", "problemi_deduplicati.txt")

all_groups = merged_groups + [[idx] for idx in singletons]

with open(output_path, "w", encoding="utf-8") as f:
    for group_num, group in enumerate(all_groups, start=1):
        n_sources = len(group)
        f.write(f"=== PROBLEMA {group_num} ({n_sources} fonte/i fusa/e) ===\n\n")
        for idx in group:
            p = problems[idx]
            f.write(f"--- Fonte: {p['source_file']} ({p['year']}) ---\n")
            f.write(p["text"])
            f.write("\n\n")
        if group_num < len(all_groups):
            f.write("-%-\n\n")

print(f"Scritti {len(all_groups)} problemi deduplicati (da {len(problems)} originali) in {output_path}")
